# ![Machine Learning Lab](banner.jpg)

# Laboratorio 8 - Actividad

## Instrucciones generales

1. Esta actividad es de carácter individual. No se permite entregar la actividad después de la fecha establecida.
2. Al responder las preguntas de las actividades, por favor marquen las respuestas con la sección a la que corresponden, por ejemplo: `## 1.1 Carga de datos`. Es preferible que esto lo hagan con secciones de MarkDown.
3. Por favor asegurarse de que el notebook entregado tenga todas las celdas ejecutadas correctamente.
4. Por favor, nombren el archivo de acuerdo con el siguiente formato `{email}_lab8.ipynb`.
5. Si tienen alguna duda, pueden escribirme a mi correo `j.rayom@uniandes.edu.co` o contactarme directamente por Teams.

---

## Objetivos

1. Construir y entrenar una Red Neuronal Convolucional (CNN) para clasificación de imágenes médicas.
2. Aplicar técnicas de preprocesamiento y aumentación de datos a imágenes.
3. Evaluar y comparar el desempeño del modelo con y sin data augmentation.

---

En esta ocasión trabajaremos con el dataset **Brain Tumor MRI Images**, que contiene imágenes de resonancia magnética cerebral clasificadas en 4 categorías: **Glioma**, **Meningioma**, **Pituitary** y **Healthy**.

Dataset: [Brain Tumor MRI Images](https://www.kaggle.com/datasets/miadul/brain-tumor-mri-dataset/data)

---

## Instrucciones

### 1. Carga y exploración de datos (10%)

1. Descarga el dataset utilizando `kagglehub`:
   ```python
   import kagglehub
   path = kagglehub.dataset_download("miadul/brain-tumor-mri-dataset")
   ```
2. Explora la estructura de carpetas del dataset y cuenta el número de imágenes por clase.
3. Visualiza al menos 3 imágenes de ejemplo de cada clase en una cuadrícula.
4. Muestra la distribución de clases en un gráfico de barras. ¿El dataset está balanceado?

---

### 2. Preprocesamiento de datos (20%)

1. Redimensiona todas las imágenes a un tamaño uniforme (por ejemplo, 150×150 píxeles).
2. Normaliza los valores de los píxeles al rango [0, 1].
3. Divide el dataset en conjuntos de entrenamiento (70%), validación (15%) y prueba (15%). Utiliza `seed = 42`.
4. Crea los datasets de TensorFlow/Keras utilizando `image_dataset_from_directory`.

---

### 3. Modelo CNN base (30%)

1. Construye una CNN con al menos 3 bloques convolucionales (Conv2D + MaxPooling2D).
2. Agrega capas Dense para la clasificación final.
3. Muestra el resumen del modelo (`model.summary()`) e indica el número total de parámetros.
4. Compila el modelo con el optimizador y función de pérdida adecuados para clasificación multiclase.
5. Entrena el modelo.
6. Grafica las curvas de pérdida y precisión (entrenamiento vs validación).

---

### 4. Data Augmentation (30%)

1. Define una estrategia de aumentación de datos que incluya al menos 3 transformaciones (rotación, volteo, zoom, etc.).
2. Visualiza el efecto de la aumentación sobre algunas imágenes de ejemplo.
3. Construye un nuevo modelo CNN que integre las capas de aumentación.
4. Entrena el modelo.
5. Grafica las curvas de aprendizaje del modelo con aumentación.
6. Compara la precisión del modelo con y sin aumentación en el conjunto de prueba.

---

### 5. Análisis de resultados (10%)

1. Genera la matriz de confusión para ambos modelos (con y sin aumentación).
2. ¿Cuáles tipos de tumor son más difíciles de clasificar?
3. Reporta la precisión (accuracy), precisión por clase y recall por clase para el mejor modelo.

---


# **1. Carga y exploración de datos**

In [ ]:
import os

os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda-12.8"

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
import numpy as np
import kagglehub
import hashlib
import shutil
import random
import os

In [ ]:
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

Descarga del conjunto de datos a utilizar haciendo uso de la libreria **kagglehub** con el fin de automatizar el proceso y hacer que sea replicable

In [ ]:
dataset_path = kagglehub.dataset_download(handle = "miadul/brain-tumor-mri-dataset", output_dir = "../datasets", force_download = True)
print(f"DATASET DOWNLOADED IN: {dataset_path}")

Exploración de la estructura de carpetas del conjunto de datos descargado anteriormente

In [ ]:
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = " " * 2 * level
    folder_name = os.path.basename(root)
    num_files = len([f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))])
    if num_files > 0:
        print(f"{indent}{folder_name}/ ({num_files} imágenes)")
    else:
        print(f"{indent}{folder_name}/")

Reorganización de la estructura de carpetas con el fin de facilitar su posterior división y carga 

In [ ]:
root_path = "../datasets"
hidden_path = os.path.join(root_path, ".complete")
former_path_classes = os.path.join(root_path, "brain_tumor_dataset")

In [ ]:
classes = ["glioma", "healthy", "meningioma", "pituitary"]

In [ ]:
if os.path.exists(hidden_path):
    shutil.rmtree(hidden_path)
    print(f"Folder '{hidden_path}' deleted")

for name in classes:
    origin = os.path.join(former_path_classes, name)
    destination = os.path.join(root_path, name)
    
    if os.path.exists(origin):
        shutil.move(origin, destination)
        print(f"☑ Class '{name}' moved to root")

if os.path.exists(former_path_classes):
    shutil.rmtree(former_path_classes)
    print(f"Intermediate folder '{former_path_classes}' deleted")

Verificación de la nueva estructura de carpetas del conjunto de datos

In [ ]:
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = " " * 2 * level
    folder_name = os.path.basename(root)
    num_files = len([f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))])
    if num_files > 0:
        print(f"{indent}{folder_name}/ ({num_files} imágenes)")
    else:
        print(f"{indent}{folder_name}/")

Eliminación de las imágenes repetidas haciendo uso de comparación mediante **códigos de hash**

In [ ]:
found_hashes = set()
counter = 0

In [ ]:
for root, dirs, files in os.walk(root_path):
    for filename in files:

        if filename.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            file_path = os.path.join(root, filename)
            
            with open(file_path, "rb") as f:
                file_hash = hashlib.md5(f.read()).hexdigest()
            
            if file_hash in found_hashes:
                print(f"Deleting duplicate: {file_path}")
                os.remove(file_path)
                counter += 1
            else:
                found_hashes.add(file_hash)

In [ ]:
print(f"NUMVER OF PRESERVED IMAGES: {len(found_hashes)}")
print(f"NUMBER OF DELETED IMAGES: {counter}")

Visualización de 4 imágenes aleatorias por cada clase del conjunto de datos

In [ ]:
data_dir = Path("../datasets") 
classes = ["glioma", "healthy", "meningioma", "pituitary"]

In [ ]:
fig, axes = plt.subplots(4, 4, figsize = (16, 17))

for row, class_name in enumerate(classes):

    class_dir = data_dir / class_name
    image_files = list(class_dir.glob("*.jpg"))
    samples = random.sample(image_files, 4)
    
    for col, img_path in enumerate(samples):

        ax = axes[row, col]
        img = tf.keras.utils.load_img(img_path, target_size = (200, 200))
        ax.imshow(img)
        
        if col == 0:
            ax.set_ylabel(class_name.upper(), fontsize = 14, fontweight = "bold")
        
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle("SAMPLE IMAGES BY CLASS", fontsize = 18, fontweight = "bold")
plt.tight_layout()
plt.show()

Conteo de la cantidad de imágenes asociadas a cada clase del conjunto de datos

In [ ]:
counts = []

for name in classes:
    class_path = os.path.join(data_dir, name)
    n_images = len(os.listdir(class_path))
    counts.append(n_images)
    print(f"Class {name}: {n_images} images")

Visualización de los conteos obtenidos anteriormente en forma de gráfico de barras

In [ ]:
plt.figure(figsize = (10, 6))

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"] 
bars = plt.bar(classes, counts, color = colors)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 20, yval, ha = "center", va = "bottom", fontweight = "bold")

plt.title("CLASSES DISTRIBUTION", fontsize = 16)
plt.xlabel("TUMOR TYPE", fontsize = 12)
plt.ylabel("# OF IMAGES", fontsize = 12)

plt.show()

|**Pregunta**|**Respuesta**|
|---|---|
|¿El conjunto de datos está balanceado?|Sí, el conjunto de datos está relativamente balanceado, ya que las cuatro clases mantienen una proporción similar (entre el 23% y el 28.5% del total). Sin embargo, es muy recomendable implementar técnicas de Data Augmentation (como rotaciones, zooms o ajustes de brillo). Esto permite compensar las ligeras diferencias numéricas entre categorías, equilibrando mejor el peso de cada clase durante el entrenamiento y mejorando la capacidad del modelo para generalizar ante casos nuevos.|

# **2. Preprocesamiento de datos**

In [ ]:
import splitfolders

División de las imágenes en conjuntos de datos para entrenamiento, validación y prueba. Se decide hacer división mediante el sistema de archivos con el fin de añadir complejidad al proceso y evitar fugas de datos

In [ ]:
splitfolders.ratio("../datasets", output = "../dataset_split", 
                   seed = SEED, ratio=(.7, .15, .15), 
                   group_prefix = None, move = False) 

Carga de las imágenes pertenecientes a las distintas clases haciendo uso de la función **image_dataset_from_directory**. Al cargar las imágenes, todas son redimensionadas a un tamaño de **150x150** píxeles y se crean batchs de **32** imágenes cada uno

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "../dataset_split/train",
    label_mode = "categorical",
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    shuffle = True  
)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    "../dataset_split/val",
    label_mode = "categorical",
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    shuffle = False 
)

In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    "../dataset_split/test",
    label_mode = "categorical",
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    shuffle = False 
)

Verificación de que la función identifica correctamente las clases de las imágenes a partir de la estructura de directorios del conjunto de datos

In [ ]:
class_names = train_ds.class_names
print(f"LOADED CLASSES: {class_names}")

Normalización de las imágenes en la totalidad de los conjuntos de datos creados del rango **[0, 255]** al rango **[0 , 1]**

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

In [ ]:
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

Optimización de los conjuntos de datos para aprovechar la carga de las imágenes en **RAM** y reducir el número de lecturas a realizar sobre el disco del sistema

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size = AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size = AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size = AUTOTUNE)

Evaluación de la estructura final de los conjuntos de datos creados

In [ ]:
print(f"BATCHES:\n")
print(f"TRAIN: {train_ds.cardinality().numpy()}")
print(f"VALIDATION: {val_ds.cardinality().numpy()}")
print(f"TEST: {test_ds.cardinality().numpy()}")

# **3. Modelo CNN base**

In [ ]:
from tensorflow.keras import layers, models

In [ ]:
IMG_HEIGHT = 150 
IMG_WIDTH = 150

num_classes = 4

In [ ]:
model = models.Sequential([

    layers.Conv2D(32, (3, 3), activation = "relu", input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation = "relu"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), activation = "relu"),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),

    layers.Dense(128, activation = "relu"),
    layers.Dense(num_classes, activation = "softmax") 
])

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer = "adam",
            loss = tf.keras.losses.CategoricalCrossentropy(from_logits = False),
            metrics = ["accuracy"])

In [ ]:
history = model.fit(
    train_ds,           
    validation_data = val_ds,
    epochs = 15,        
    verbose = 1
)

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs_range = range(len(acc))

In [ ]:
plt.figure(figsize = (12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label = "TRAINING")
plt.plot(epochs_range, val_acc, label = "VALIDATION")
plt.title("ACCURACY")
plt.xlabel("EPOCHS")
plt.legend(loc = "lower right")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label = "TRAINING")
plt.plot(epochs_range, val_loss, label = "VALIDATION")
plt.title("LOSS")
plt.xlabel("EPOCHS")
plt.legend(loc = "upper right")
plt.grid(True)

plt.show()